# LangGraph Dialogue System Simulation

This notebook runs the new `maestro_langgraph` module for testing.

**Features:**
- No hardcoded responses - all generated by LLM
- DuckDuckGo web search for factual questions
- Emotion detection via LLM
- **Busy/Problem context checking** (like original MAESTRO)
- Supports both **Ollama** and **LM Studio** backends

**Backends:**
- **Ollama**: Uses `qwen2:0.5b` (0.5B params) - fastest
- **LM Studio**: Uses any loaded model via OpenAI-compatible API

**Requirements:**
- `uv sync --extra langgraph`
- Either Ollama or LM Studio running locally

## 1. Setup

In [37]:
import sys
from pathlib import Path

# Add maestro_langgraph to path
notebook_dir = Path.cwd()
module_dir = notebook_dir / "maestro_langgraph"

if module_dir.exists():
    sys.path.insert(0, str(notebook_dir))
    print(f"Added to path: {notebook_dir}")
else:
    print(f"Module not found at {module_dir}")
    print(f"Current dir: {notebook_dir}")

Added to path: /Users/tomaszkoczar/Desktop/Projects/Robo/rooted/src/maestro_langgraph


In [38]:
# Check available backends
import subprocess
import urllib.request

def check_ollama():
    """Check if Ollama is running."""
    try:
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
        print("✓ Ollama available")
        print(result.stdout)
        return True
    except Exception as e:
        print(f"✗ Ollama not available: {e}")
        return False

def check_lmstudio():
    """Check if LM Studio server is running."""
    try:
        req = urllib.request.Request("http://localhost:1234/v1/models", method="GET")
        with urllib.request.urlopen(req, timeout=5) as response:
            print("✓ LM Studio available at localhost:1234")
            return True
    except Exception as e:
        print(f"✗ LM Studio not available: {e}")
        return False

print("Checking available backends...")
print("-" * 50)
ollama_ok = check_ollama()
print()
lmstudio_ok = check_lmstudio()
print("-" * 50)

if ollama_ok:
    print("\n→ To use Ollama: BACKEND = 'ollama'")
if lmstudio_ok:
    print("→ To use LM Studio: BACKEND = 'lmstudio'")
if not ollama_ok and not lmstudio_ok:
    print("\n⚠️  No backend available!")
    print("   Start Ollama: ollama serve")
    print("   Or start LM Studio server")

Checking available backends...
--------------------------------------------------
✗ Ollama not available: [Errno 2] No such file or directory: 'ollama'

✓ LM Studio available at localhost:1234
--------------------------------------------------
→ To use LM Studio: BACKEND = 'lmstudio'


## 2. Import Module

In [ ]:
from maestro_langgraph.state import DialogueState, Intent, Emotion, RobotStatus, get_prosody_for_emotion
from maestro_langgraph.chains import DialogueChains
from maestro_langgraph.graph import process_message, set_chains, build_dialogue_graph

print("✓ Imported maestro_langgraph module")
print(f"  Intents: {[i.value for i in Intent]}")
print(f"  Emotions: {[e.value for e in Emotion]}")
print(f"  Robot Status: {[s.value for s in RobotStatus]}")

## 3. Initialize Chains

Choose your backend below:
- `"ollama"` - Uses Ollama with specified model (default: qwen2:0.5b)
- `"lmstudio"` - Uses LM Studio with currently loaded model

In [40]:
# ============================================
# CONFIGURE YOUR BACKEND HERE
# ============================================
BACKEND = "lmstudio"  # "ollama" or "lmstudio"
MODEL = "qwen3:0.5b"  # For Ollama only (LM Studio uses loaded model)
# ============================================

try:
    chains = DialogueChains(
        model_name=MODEL,
        backend=BACKEND,
    )
    set_chains(chains)
    print(f"✓ Initialized DialogueChains")
    print(f"  Backend: {BACKEND}")
    if BACKEND == "ollama":
        print(f"  Model: {MODEL}")
    else:
        print(f"  Model: (using LM Studio's loaded model)")
    print(f"  DuckDuckGo search: enabled")
except Exception as e:
    print(f"❌ Failed to initialize: {e}")
    if BACKEND == "ollama":
        print(f"\nMake sure Ollama is running and model '{MODEL}' is pulled")
        print("  ollama serve")
        print(f"  ollama pull {MODEL}")
    else:
        print("\nMake sure LM Studio server is running on port 1234")
        print("  1. Open LM Studio")
        print("  2. Load a model")
        print("  3. Start the server (Developer tab)")

✓ Initialized DialogueChains
  Backend: lmstudio
  Model: (using LM Studio's loaded model)
  DuckDuckGo search: enabled


## 4. Test Individual Components

In [41]:
# Test intent classification
test_messages = [
    "Hello there!",
    "What is photosynthesis?",
    "Goodbye!",
    "Yes, that sounds good",
    "Tell me about soil moisture",
]

print("Testing Intent Classification:")
print("-" * 50)
for msg in test_messages:
    intent = chains.classify_intent(msg)
    print(f"  '{msg}' → {intent}")

Testing Intent Classification:
--------------------------------------------------
  'Hello there!' → greeting
  'What is photosynthesis?' → question
  'Goodbye!' → farewell
  'Yes, that sounds good' → acknowledgment
  'Tell me about soil moisture' → question


In [42]:
# Test web search
print("Testing DuckDuckGo Search:")
print("-" * 50)

query = "What is photosynthesis?"
print(f"Query: {query}")
results = chains.search_web(query)
print(f"Results (first 500 chars):\n{results[:500]}...")

Testing DuckDuckGo Search:
--------------------------------------------------
Query: What is photosynthesis?
Results (first 500 chars):
What is photosynthesis ? ... Photosynthesis is the process plants, algae and some bacteria use to turn sunlight, carbon dioxide and water into sugar ... What is Photosynthesis ... This process is called photosynthesis and is performed by all plants, algae, and even some microorganisms. What Is Photosynthesis ? Photosynthesis is the process by which green plants make food for themselves-and, indirectly, for all animals, including ... Order custom essay What Is Photosynthesis And How It Works with...


In [29]:
# Test response generation
print("Testing Response Generation:")
print("-" * 50)

response = chains.generate_response(
    message="Hello, how are you?",
    emotion="happy",
    history=[]
)
print(f"Response: {response}")

Testing Response Generation:
--------------------------------------------------
Response: Hello! Welcome to the Plant Doctor. How can I assist you today?


In [44]:
# Test emotion detection
print("Testing Emotion Detection:")
print("-" * 50)

emotion = chains.detect_emotion(
    user_message="Kurwa!",
    robot_response="Kurwa"
)
print(f"Detected emotion: {emotion}")
print(f"Prosody: {get_prosody_for_emotion(emotion)}")

Testing Emotion Detection:
--------------------------------------------------
Detected emotion: anger
Prosody: (180, 200, 55)


## 5. Run Full Dialogue Graph

In [45]:
def chat(message: str, history: list = None, voice_emotion: str = "neutral"):
    """Process a message through the dialogue graph."""
    print(f"\n{'='*60}")
    print(f"Human: {message}")
    print(f"{'='*60}")

    result = process_message(
        message=message,
        voice_emotion=voice_emotion,
        history=history or [],
    )

    print(f"\nIntent: {result.get('intent', 'unknown')}")
    if result.get('search_context'):
        print(f"Search: Yes (found context)")
    print(f"\n🤖 Plantroid: {result.get('response', '')}")
    print(f"   Emotion: {result.get('response_emotion', 'neutral')}")
    print(f"   Prosody: {result.get('prosody', (150, 100, 45))}")

    return result

In [46]:
# Test greeting
result = chat("Hello Plantroid!")


Human: Hello Plantroid!

Intent: greeting

🤖 Plantroid: Hi there! 😊 It's great to meet you—let's get those plants thriving together!
   Emotion: happy
   Prosody: (160, 140, 65)


In [47]:
# Test question (triggers web search)
result = chat("What is the best soil for tomatoes?")


Human: What is the best soil for tomatoes?

Intent: question
Search: Yes (found context)

🤖 Plantroid: Hey there! For tomatoes, aim for a well‑draining mix with about 30‑40 % organic matter (like compost or coco‑coir), plus perlite or vermiculite to keep the soil airy and prevent waterlogging. This blend gives the roots plenty of nutrients while keeping them from getting soggy. Happy planting!
   Emotion: happy
   Prosody: (160, 140, 65)


In [48]:
# Test factual question with search
result = chat("Who is Albert Einstein?")


Human: Who is Albert Einstein?

Intent: question
Search: Yes (found context)

🤖 Plantroid: Hi there! Albert Einstein was a famous physicist who developed the theory of relativity, but I'm here to help with plant care and soil questions—what would you like to know? 😊
   Emotion: happy
   Prosody: (160, 140, 65)


In [49]:
# Test farewell
result = chat("Goodbye!")


Human: Goodbye!

Intent: farewell

🤖 Plantroid: Have a wonderful day! 😊
   Emotion: happy
   Prosody: (160, 140, 65)


## 6. Multi-turn Conversation

In [50]:
# Multi-turn conversation with history
history = []

print("\n" + "="*60)
print("MULTI-TURN CONVERSATION")
print("="*60)

messages = [
    "Hi there!",
    "What's your name?",
    "Can you help me with my plants?",
    "What is the ideal temperature for succulents?",
    "Thanks for your help!",
]

for msg in messages:
    result = process_message(
        message=msg,
        history=history,
    )

    print(f"\nHuman: {msg}")
    print(f"🤖 Plantroid: {result.get('response', '')}")

    # Update history
    history = result.get('conversation_history', [])


MULTI-TURN CONVERSATION

Human: Hi there!
🤖 Plantroid: Hi there! I'm Plantroid, your friendly plant-care companion—what would you like to know about caring for your green friends today? 😊

Human: What's your name?
🤖 Plantroid: Hi there! I'm Plantroid—your friendly robot companion for all things plant care. 😊 How can I help you today?

Human: Can you help me with my plants?
🤖 Plantroid: Sure! I'd love to help you with your plants—just let me know what you need! 😊

Human: What is the ideal temperature for succulents?
🤖 Plantroid: Ideal temperatures for succulents are typically between 60°F and 75°F (15°C to 24°C) during the day, with nighttime temperatures not dropping below 40°F (4°C). Bring them indoors when it gets colder to keep them thriving! 🌵

Human: Thanks for your help!
🤖 Plantroid: You're very welcome! Happy to help—keep those succulents thriving! 😊


In [51]:
# View conversation history
print("Conversation History:")
print("-" * 40)
for role, msg in history:
    print(f"{role}: {msg}")

Conversation History:
----------------------------------------
Human: Hi there!
Plantroid: Hi there! I'm Plantroid, your friendly plant-care companion—what would you like to know about caring for your green friends today? 😊
Human: What's your name?
Plantroid: Hi there! I'm Plantroid—your friendly robot companion for all things plant care. 😊 How can I help you today?
Human: Can you help me with my plants?
Plantroid: Sure! I'd love to help you with your plants—just let me know what you need! 😊
Human: What is the ideal temperature for succulents?
Plantroid: Ideal temperatures for succulents are typically between 60°F and 75°F (15°C to 24°C) during the day, with nighttime temperatures not dropping below 40°F (4°C). Bring them indoors when it gets colder to keep them thriving! 🌵
Human: Thanks for your help!
Plantroid: You're very welcome! Happy to help—keep those succulents thriving! 😊


In [ ]:
# Test FREE state - normal conversation flow
print("="*60)
print("TEST: Robot is FREE (normal flow)")
print("="*60)

result = process_message(
    message="What's the best way to water succulents?",
    robot_busy=False,
    notifications={},  # No problems
)

print(f"Robot Status: {result.get('robot_status')}")
print(f"Intent: {result.get('intent')}")
print(f"Response: {result.get('response')}")
print(f"Emotion: {result.get('response_emotion')}")

In [ ]:
# Test PROBLEM state - sensor notifications pending
print("="*60)
print("TEST: Robot has PROBLEM notifications")
print("="*60)

# Simulate sensor notifications (like original MAESTRO)
notifications = {
    "soil_moisture": ["low", "high"],
    "temperature": ["28°C", "medium"],
}

result = process_message(
    message="Hi there!",
    robot_busy=False,
    notifications=notifications,  # Has problems to announce!
)

print(f"Robot Status: {result.get('robot_status')}")
print(f"Response: {result.get('response')}")
print(f"Emotion: {result.get('response_emotion')} (should be 'sad' for concern)")
print(f"Should end early: {result.get('should_end_early')} (False = allow follow-up)")

In [ ]:
# Test BUSY state - robot is busy with a task
print("="*60)
print("TEST: Robot is BUSY")
print("="*60)

result = process_message(
    message="Hello, can you help me?",
    robot_busy=True,  # Robot is busy!
    notifications={},
)

print(f"Robot Status: {result.get('robot_status')}")
print(f"Response: {result.get('response')}")
print(f"Emotion: {result.get('response_emotion')}")
print(f"Should end early: {result.get('should_end_early')}")

In [ ]:
# Visualize the dialogue graph structure
try:
    from langgraph.graph import StateGraph

    print("LangGraph Dialogue Flow (with Busy/Problem Check):")
    print("="*60)
    print("""
    START
      │
      ▼
    ┌─────────────────┐
    │  check_context  │  ← Check busy/problem status
    └────────┬────────┘
             │
      ┌──────┴──────┐
      │             │
      ▼             ▼
   [BUSY]       [FREE/PROBLEM]
      │             │
      │             ▼
      │     ┌─────────────────┐
      │     │ classify_intent │  ← LLM classifies intent
      │     └────────┬────────┘
      │              │
      │              ▼
      │     ┌─────────────────┐
      │     │  check_search   │  ← DuckDuckGo if needed
      │     └────────┬────────┘
      │              │
      └──────┬───────┘
             │
             ▼
    ┌─────────────────┐
    │generate_response│  ← LLM generates response
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │determine_emotion│  ← Emotion based on context
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │ update_history  │  ← Store conversation
    └────────┬────────┘
             │
             ▼
           END
             
    Context-aware emotions:
    - BUSY → neutral (apologetic)
    - PROBLEM → sad (concerned)
    - FREE → LLM-determined
    """)

except Exception as e:
    print(f"Could not visualize: {e}")

## 7. Interactive Chat

In [22]:
# Interactive chat loop
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("Widgets not available. Using text input.")

if WIDGETS_AVAILABLE:
    output = widgets.Output()
    text_input = widgets.Text(
        placeholder='Talk to Plantroid...',
        layout=widgets.Layout(width='70%')
    )
    send_btn = widgets.Button(description='Send', button_style='primary')
    clear_btn = widgets.Button(description='Clear', button_style='warning')

    chat_history = []

    def on_send(_):
        global chat_history
        if text_input.value:
            with output:
                result = process_message(
                    message=text_input.value,
                    history=chat_history,
                )
                print(f"\nYou: {text_input.value}")
                print(f"🤖: {result.get('response', '')} [{result.get('response_emotion', '')}]")
                chat_history = result.get('conversation_history', [])
            text_input.value = ''

    def on_clear(_):
        global chat_history
        chat_history = []
        with output:
            clear_output()
            print("Chat cleared.")

    send_btn.on_click(on_send)
    clear_btn.on_click(on_clear)
    text_input.on_submit(lambda _: on_send(None))

    print("💬 Chat with Plantroid (LangGraph)")
    print("All responses generated by LLM - no hardcoded flows!")
    display(widgets.HBox([text_input, send_btn, clear_btn]))
    display(output)

💬 Chat with Plantroid (LangGraph)
All responses generated by LLM - no hardcoded flows!


/var/folders/bb/573j0mcn4ds4f9crvgk_hmf00000gn/T/ipykernel_17891/1732188571.py:43: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(lambda _: on_send(None))


Output()

## 8. Visualize Graph

In [23]:
# Visualize the dialogue graph structure
try:
    from langgraph.graph import StateGraph

    print("LangGraph Dialogue Flow:")
    print("="*50)
    print("""
    START
      │
      ▼
    ┌─────────────────┐
    │ classify_intent │  ← LLM classifies user intent
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │  check_search   │  ← DuckDuckGo if needed
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │generate_response│  ← LLM generates response
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │determine_emotion│  ← LLM detects emotion
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │ update_history  │  ← Store conversation
    └────────┬────────┘
             │
             ▼
           END
    """)

except Exception as e:
    print(f"Could not visualize: {e}")

LangGraph Dialogue Flow:

    START
      │
      ▼
    ┌─────────────────┐
    │ classify_intent │  ← LLM classifies user intent
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │  check_search   │  ← DuckDuckGo if needed
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │generate_response│  ← LLM generates response
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │determine_emotion│  ← LLM detects emotion
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │ update_history  │  ← Store conversation
    └────────┬────────┘
             │
             ▼
           END
    


## 9. Performance Test

In [ ]:
import time

# Measure response time
test_messages = [
    "Hello!",
    "How are you?",
    "What is the weather like?",
]

print("Performance Test:")
print("-" * 50)

times = []
for msg in test_messages:
    start = time.time()
    result = process_message(message=msg)
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"'{msg}' → {elapsed:.2f}s")

print(f"\nAverage response time: {sum(times)/len(times):.2f}s")
print(f"Backend: {BACKEND}")
if BACKEND == "ollama":
    print(f"Model: {MODEL}")